# Phase 6 — Data Cleaning & Preparation

## Objective

The objective of this phase is to transform the raw Olist datasets into clean, consistent, and analysis-ready datasets while preserving valid business information.

The cleaning process is based on the data quality issues identified during Phase 5 — Data Profiling.

## Cleaning Principles

- Do not remove valid business records unnecessarily.
- Handle missing values according to their business meaning.
- Preserve legitimate one-to-many relationships.
- Standardize data types and categorical values.
- Treat duplicates according to the role of each dataset.
- Validate business rules before modifying data.
- Document important cleaning decisions.
- Keep the raw datasets unchanged.

## Main Cleaning Areas

1. Missing values
2. Data types
3. Date and time fields
4. Categorical and text consistency
5. Duplicates
6. Invalid values
7. Outliers
8. Referential integrity
9. Final validation

## Output

The final output of this phase will be cleaned, analysis-ready datasets derived from the raw Olist datasets.

## Issues Identified During Data Profiling

The following issues identified in Phase 5 will be addressed during this phase:

- Missing values in selected columns.
- Missing delivery dates depending on order status.
- Missing review comments.
- Missing product attributes.
- Duplicate records in the geolocation dataset.
- Multiple customer IDs associated with the same customer unique ID.
- Multiple payment records associated with some orders.
- Multiple review records associated with some orders.
- Data type inconsistencies, particularly date columns stored as strings.
- Categorical/value consistency issues.
- Business-rule exceptions identified during validation.
- Skewed numerical variables and potential outliers requiring business-aware treatment.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW_DATA_PATH = Path("../Data/Raw")
CLEAN_DATA_PATH = Path("../Data/Cleaned")

CLEAN_DATA_PATH.mkdir(parents=True, exist_ok=True)

print("Raw data path:", RAW_DATA_PATH.resolve())
print("Cleaned data path:", CLEAN_DATA_PATH.resolve())

Raw data path: E:\ShopSphere_Analysis\Data\Raw
Cleaned data path: E:\ShopSphere_Analysis\Data\Cleaned


In [2]:
files = sorted(RAW_DATA_PATH.glob("*.csv"))

print(f"Found {len(files)} CSV files")

for file in files:
    print(file.name)

Found 11 CSV files
olist_closed_deals_dataset.csv
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_marketing_qualified_leads_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


## Step 2 — Load Raw Datasets

All raw CSV files are loaded into a dictionary so that they can be accessed systematically during the cleaning process.

The raw files are not modified directly.

In [3]:
datasets = {}

for file in files:
    dataset_name = file.stem.replace("olist_", "").replace("_dataset", "")
    datasets[dataset_name] = pd.read_csv(file)

print(f"Loaded {len(datasets)} datasets:\n")

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

Loaded 11 datasets:

closed_deals: 842 rows × 14 columns
customers: 99,441 rows × 5 columns
geolocation: 1,000,163 rows × 5 columns
marketing_qualified_leads: 8,000 rows × 4 columns
order_items: 112,650 rows × 7 columns
order_payments: 103,886 rows × 5 columns
order_reviews: 99,224 rows × 7 columns
orders: 99,441 rows × 8 columns
products: 32,951 rows × 9 columns
sellers: 3,095 rows × 4 columns
product_category_name_translation: 71 rows × 2 columns


In [4]:
datasets["orders"].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
datasets["orders"].info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [6]:
# Step 2: Initial Data Type Inspection

for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"DATASET: {name}")
    print(f"{'=' * 60}")
    print(df.dtypes)


DATASET: closed_deals
mql_id                               str
seller_id                            str
sdr_id                               str
sr_id                                str
won_date                             str
business_segment                     str
lead_type                            str
lead_behaviour_profile               str
has_company                       object
has_gtin                          object
average_stock                        str
business_type                        str
declared_product_catalog_size    float64
declared_monthly_revenue         float64
dtype: object

DATASET: customers
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

DATASET: geolocation
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geoloca

In [7]:
# Check the number of missing values

for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    print(f"\n{name}")
    print(missing)


closed_deals
business_segment                   1
lead_type                          6
lead_behaviour_profile           177
has_company                      779
has_gtin                         778
average_stock                    776
business_type                     10
declared_product_catalog_size    773
dtype: int64

customers
Series([], dtype: int64)

geolocation
Series([], dtype: int64)

marketing_qualified_leads
origin    60
dtype: int64

order_items
Series([], dtype: int64)

order_payments
Series([], dtype: int64)

order_reviews
review_comment_title      87656
review_comment_message    58247
dtype: int64

orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

products
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm             

## Step 3: Data Type Standardization

Convert date fields to datetime format and standardize inconsistent data types
to prepare the datasets for analysis and downstream transformations.

In [8]:
# Step 3: Standardize date columns

date_columns = {
    "closed_deals": [
        "won_date"
    ],
    "marketing_qualified_leads": [
        "first_contact_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "order_reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ],
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
}

for dataset_name, columns in date_columns.items():
    df = datasets[dataset_name]

    for column in columns:
        df[column] = pd.to_datetime(df[column], errors="coerce")

print("Date columns standardized successfully.")

Date columns standardized successfully.


In [9]:
for dataset_name, columns in date_columns.items():
    print(f"\n{dataset_name}")

    df = datasets[dataset_name]

    for column in columns:
        print(f"{column}: {df[column].dtype}")


closed_deals
won_date: datetime64[us]

marketing_qualified_leads
first_contact_date: datetime64[us]

order_items
shipping_limit_date: datetime64[us]

order_reviews
review_creation_date: datetime64[us]
review_answer_timestamp: datetime64[us]

orders
order_purchase_timestamp: datetime64[us]
order_approved_at: datetime64[us]
order_delivered_carrier_date: datetime64[us]
order_delivered_customer_date: datetime64[us]
order_estimated_delivery_date: datetime64[us]


In [10]:
# Step 4: Inspect potentially inconsistent numeric columns

print("closed_deals - average_stock")
print(datasets["closed_deals"]["average_stock"].value_counts(dropna=False).head(20))

print("\nData type:")
print(datasets["closed_deals"]["average_stock"].dtype)

closed_deals - average_stock
average_stock
NaN        776
5-20        22
50-200      15
1-5         10
20-50        8
200+         7
unknown      4
Name: count, dtype: int64

Data type:
str


In [11]:
# Check non-numeric values

average_stock_numeric = pd.to_numeric(
    datasets["closed_deals"]["average_stock"],
    errors="coerce"
)

non_numeric = (
    datasets["closed_deals"]["average_stock"].notna()
    & average_stock_numeric.isna()
)

print("Non-numeric values:")
print(datasets["closed_deals"].loc[non_numeric, "average_stock"].unique())

print("\nNumber of non-numeric values:", non_numeric.sum())

Non-numeric values:
<StringArray>
['20-50', '1-5', '5-20', '200+', '50-200', 'unknown']
Length: 6, dtype: str

Number of non-numeric values: 66


In [12]:
# Step 6.x: Inspect declared_product_catalog_size

df = datasets["closed_deals"]

print(df["declared_product_catalog_size"].value_counts(dropna=False).sort_index())
print("\nData type:")
print(df["declared_product_catalog_size"].dtype)

print("\nDescriptive statistics:")
print(df["declared_product_catalog_size"].describe())

declared_product_catalog_size
1.0         1
2.0         1
4.0         2
5.0         1
10.0        3
12.0        1
15.0        2
20.0        4
22.0        1
30.0        2
40.0        2
45.0        1
47.0        1
50.0        7
70.0        2
75.0        1
80.0        1
85.0        1
100.0       9
120.0       2
132.0       1
200.0       2
300.0       5
305.0       1
400.0       4
500.0       2
550.0       1
600.0       1
700.0       1
800.0       1
1000.0      3
1200.0      1
2000.0      1
NaN       773
Name: count, dtype: int64

Data type:
float64

Descriptive statistics:
count      69.000000
mean      233.028986
std       352.380558
min         1.000000
25%        30.000000
50%       100.000000
75%       300.000000
max      2000.000000
Name: declared_product_catalog_size, dtype: float64


In [13]:
# Step 6.x: Inspect declared_monthly_revenue

df = datasets["closed_deals"]

print(df["declared_monthly_revenue"].value_counts(dropna=False).sort_index())

print("\nData type:")
print(df["declared_monthly_revenue"].dtype)

print("\nDescriptive statistics:")
print(df["declared_monthly_revenue"].describe())

print("\nZero values:")
print((df["declared_monthly_revenue"] == 0).sum())

print("\nNegative values:")
print((df["declared_monthly_revenue"] < 0).sum())

declared_monthly_revenue
0.0           797
6.0             1
1000.0          1
4000.0          1
5000.0          2
6000.0          1
8000.0          1
10000.0         3
15000.0         2
20000.0         3
25000.0         3
30000.0         3
40000.0         1
50000.0         2
60000.0         2
100000.0        5
120000.0        2
130000.0        1
150000.0        1
180000.0        1
200000.0        1
210000.0        1
250000.0        2
300000.0        2
500000.0        1
8000000.0       1
50000000.0      1
Name: count, dtype: int64

Data type:
float64

Descriptive statistics:
count    8.420000e+02
mean     7.337768e+04
std      1.744799e+06
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      5.000000e+07
Name: declared_monthly_revenue, dtype: float64

Zero values:
797

Negative values:
0


In [14]:
# Step 6.x: Inspect has_company

df = datasets["closed_deals"]

print(df["has_company"].value_counts(dropna=False))

print("\nData type:")
print(df["has_company"].dtype)

print("\nUnique non-null values:")
print(df["has_company"].dropna().unique())

has_company
NaN      779
True      58
False      5
Name: count, dtype: int64

Data type:
object

Unique non-null values:
[True False]


In [15]:
# Step 6.x: Inspect has_gtin

df = datasets["closed_deals"]

print(df["has_gtin"].value_counts(dropna=False))

print("\nData type:")
print(df["has_gtin"].dtype)

print("\nUnique non-null values:")
print(df["has_gtin"].dropna().unique())

has_gtin
NaN      778
True      54
False     10
Name: count, dtype: int64

Data type:
object

Unique non-null values:
[True False]


In [16]:
# Step 6.x: Inspect business_type

df = datasets["closed_deals"]

print(df["business_type"].value_counts(dropna=False))

print("\nData type:")
print(df["business_type"].dtype)

print("\nUnique non-null values:")
print(df["business_type"].dropna().unique())

business_type
reseller        587
manufacturer    242
NaN              10
other             3
Name: count, dtype: int64

Data type:
str

Unique non-null values:
<StringArray>
['reseller', 'manufacturer', 'other']
Length: 3, dtype: str


In [17]:
# Step 6.x: Inspect business_segment

df = datasets["closed_deals"]

print(df["business_segment"].value_counts(dropna=False))

print("\nData type:")
print(df["business_segment"].dtype)

print("\nUnique non-null values:")
print(df["business_segment"].dropna().unique())

business_segment
home_decor                         105
health_beauty                       93
car_accessories                     77
household_utilities                 71
construction_tools_house_garden     69
audio_video_electronics             64
computers                           34
pet                                 30
food_supplement                     28
food_drink                          26
sports_leisure                      25
bed_bath_table                      22
bags_backpacks                      22
toys                                20
fashion_accessories                 19
home_office_furniture               14
stationery                          13
phone_mobile                        13
small_appliances                    12
handcrafted                         12
baby                                10
music_instruments                    9
books                                9
watches                              8
jewerly                              8
home_app

In [18]:
# Step 6.x: Inspect lead_type

df = datasets["closed_deals"]

print(df["lead_type"].value_counts(dropna=False))

print("\nData type:")
print(df["lead_type"].dtype)

print("\nUnique non-null values:")
print(df["lead_type"].dropna().unique())

lead_type
online_medium      332
online_big         126
industry           123
offline            104
online_small        77
online_beginner     57
online_top          14
NaN                  6
other                3
Name: count, dtype: int64

Data type:
str

Unique non-null values:
<StringArray>
[  'online_medium',        'industry',      'online_big',    'online_small',
         'offline',      'online_top', 'online_beginner',           'other']
Length: 8, dtype: str


In [19]:
# Step 6.x: Inspect lead_behaviour_profile

df = datasets["closed_deals"]

print(df["lead_behaviour_profile"].value_counts(dropna=False))

print("\nData type:")
print(df["lead_behaviour_profile"].dtype)

print("\nUnique non-null values:")
print(df["lead_behaviour_profile"].dropna().unique())

lead_behaviour_profile
cat            407
NaN            177
eagle          123
wolf            95
shark           24
cat, wolf        8
eagle, wolf      3
eagle, cat       3
shark, cat       1
shark, wolf      1
Name: count, dtype: int64

Data type:
str

Unique non-null values:
<StringArray>
[        'cat',       'eagle',        'wolf',       'shark',   'cat, wolf',
 'eagle, wolf',  'shark, cat',  'eagle, cat', 'shark, wolf']
Length: 9, dtype: str


In [22]:
# Step 12: Categorical & Value Consistency - Remaining Columns

categorical_checks = {
    "customers": [
        "customer_city",
        "customer_state"
    ],
    
    "geolocation": [
        "geolocation_city",
        "geolocation_state"
    ],
    
    "marketing_qualified_leads": [
        "origin"
    ],
    
    "orders": [
        "order_status"
    ],
    
    "order_payments": [
        "payment_type"
    ],
    
    "order_reviews": [
        "review_comment_title",
        "review_comment_message"
    ],
    
    "products": [
        "product_category_name"
    ],
    
    "sellers": [
        "seller_city",
        "seller_state"
    ],
    
    "product_category_name_translation": [
        "product_category_name",
        "product_category_name_english"
    ]
}


for dataset_name, columns in categorical_checks.items():

    df = datasets[dataset_name]

    print("\n" + "=" * 70)
    print(f"DATASET: {dataset_name}")
    print("=" * 70)

    for col in columns:

        print(f"\n--- {col} ---")

        print("Data type:")
        print(df[col].dtype)

        print("\nMissing values:")
        print(df[col].isna().sum())

        print("\nUnique values:")
        print(df[col].nunique(dropna=True))

        print("\nTop values:")
        print(df[col].value_counts(dropna=False).head(20))


DATASET: customers

--- customer_city ---
Data type:
str

Missing values:
0

Unique values:
4119

Top values:
customer_city
sao paulo                15540
rio de janeiro            6882
belo horizonte            2773
brasilia                  2131
curitiba                  1521
campinas                  1444
porto alegre              1379
salvador                  1245
guarulhos                 1189
sao bernardo do campo      938
niteroi                    849
santo andre                797
osasco                     746
santos                     713
goiania                    692
sao jose dos campos        691
fortaleza                  654
sorocaba                   633
recife                     613
florianopolis              570
Name: count, dtype: int64

--- customer_state ---
Data type:
str

Missing values:
0

Unique values:
27

Top values:
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE  

In [23]:
# Step 12: String Formatting Consistency Check

for dataset_name, columns in categorical_checks.items():

    df = datasets[dataset_name]

    print("\n" + "=" * 70)
    print(f"DATASET: {dataset_name}")
    print("=" * 70)

    for col in columns:

        series = df[col].dropna().astype(str)

        leading_trailing_spaces = (series != series.str.strip()).sum()
        uppercase_values = (series != series.str.lower()).sum()

        print(f"\n{col}")
        print(f"Leading/trailing spaces : {leading_trailing_spaces}")
        print(f"Uppercase/mixed case    : {uppercase_values}")


DATASET: customers

customer_city
Leading/trailing spaces : 0
Uppercase/mixed case    : 0

customer_state
Leading/trailing spaces : 0
Uppercase/mixed case    : 99441

DATASET: geolocation

geolocation_city
Leading/trailing spaces : 1
Uppercase/mixed case    : 0

geolocation_state
Leading/trailing spaces : 0
Uppercase/mixed case    : 1000163

DATASET: marketing_qualified_leads

origin
Leading/trailing spaces : 0
Uppercase/mixed case    : 0

DATASET: orders

order_status
Leading/trailing spaces : 0
Uppercase/mixed case    : 0

DATASET: order_payments

payment_type
Leading/trailing spaces : 0
Uppercase/mixed case    : 0

DATASET: order_reviews

review_comment_title
Leading/trailing spaces : 1998
Uppercase/mixed case    : 8862

review_comment_message
Leading/trailing spaces : 9451
Uppercase/mixed case    : 34839

DATASET: products

product_category_name
Leading/trailing spaces : 0
Uppercase/mixed case    : 0

DATASET: sellers

seller_city
Leading/trailing spaces : 0
Uppercase/mixed case  

In [26]:
# Find the geolocation city with leading/trailing spaces

geo = datasets["geolocation"]

mask = geo["geolocation_city"].astype(str) != geo["geolocation_city"].astype(str).str.strip()

geo.loc[mask, "geolocation_city"]

670078    salvador 
Name: geolocation_city, dtype: str

In [27]:
# Identify the geolocation city containing leading/trailing whitespace

geo = datasets["geolocation"]

mask = (
    geo["geolocation_city"].notna()
    & (geo["geolocation_city"] != geo["geolocation_city"].str.strip())
)

print(geo.loc[mask, "geolocation_city"].to_string(index=False))

salvador 


In [28]:
# Check the business_segment categories
closed = datasets["closed_deals"]

print(closed["business_segment"].value_counts(dropna=False).to_string())

business_segment
home_decor                         105
health_beauty                       93
car_accessories                     77
household_utilities                 71
construction_tools_house_garden     69
audio_video_electronics             64
computers                           34
pet                                 30
food_supplement                     28
food_drink                          26
sports_leisure                      25
bed_bath_table                      22
bags_backpacks                      22
toys                                20
fashion_accessories                 19
home_office_furniture               14
stationery                          13
phone_mobile                        13
small_appliances                    12
handcrafted                         12
baby                                10
music_instruments                    9
books                                9
watches                              8
jewerly                              8
home_app

In [29]:
# Analyze lead_type

df = datasets["closed_deals"]

col = "lead_type"

print("Data type:")
print(df[col].dtype)

print("\nMissing values:")
print(df[col].isna().sum())

print("\nUnique values:")
print(df[col].nunique(dropna=True))

print("\nValue counts:")
print(df[col].value_counts(dropna=False))

Data type:
str

Missing values:
6

Unique values:
8

Value counts:
lead_type
online_medium      332
online_big         126
industry           123
offline            104
online_small        77
online_beginner     57
online_top          14
NaN                  6
other                3
Name: count, dtype: int64


In [30]:
# Step: Final categorical consistency check

categorical_issues = []

for name, df in datasets.items():

    categorical_cols = df.select_dtypes(include=["str", "object"]).columns

    for col in categorical_cols:

        series = df[col].dropna().astype(str)

        # Check whitespace
        leading_trailing = (series != series.str.strip()).sum()

        # Check uppercase / mixed case
        uppercase_mixed = (series != series.str.lower()).sum()

        # Check empty strings
        empty_strings = (series.str.strip() == "").sum()

        # Check number of unique values
        unique_count = series.nunique()

        if leading_trailing > 0 or uppercase_mixed > 0 or empty_strings > 0:

            categorical_issues.append({
                "dataset": name,
                "column": col,
                "unique_values": unique_count,
                "leading_trailing_spaces": leading_trailing,
                "uppercase_mixed_case": uppercase_mixed,
                "empty_strings": empty_strings
            })

categorical_issues_df = pd.DataFrame(categorical_issues)

categorical_issues_df

,dataset,column,unique_values,leading_trailing_spaces,uppercase_mixed_case,empty_strings
0,closed_deals,has_company,2,0,63,0
1,closed_deals,has_gtin,2,0,64,0
2,customers,customer_state,27,0,99441,0
3,geolocation,geolocation_city,8011,1,0,0
4,geolocation,geolocation_state,27,0,1000163,0
5,order_reviews,review_comment_title,4527,1998,8862,2
6,order_reviews,review_comment_message,36159,9451,34839,27
7,sellers,seller_state,23,0,3095,0


In [31]:
# Step: Final categorical consistency check

for dataset_name, df in datasets.items():

    print("\n" + "=" * 70)
    print(f"DATASET: {dataset_name}")
    print("=" * 70)

    categorical_cols = df.select_dtypes(include=["str", "object"]).columns

    for col in categorical_cols:

        # Ignore ID columns and datetime columns
        if col.endswith("_id") or "timestamp" in col or "date" in col:
            continue

        series = df[col].dropna().astype(str)

        if len(series) == 0:
            continue

        normalized = series.str.strip().str.lower()

        # Values that differ only because of spaces/case
        variants = (
            pd.DataFrame({
                "original": series,
                "normalized": normalized
            })
            .drop_duplicates()
            .groupby("normalized")["original"]
            .nunique()
        )

        inconsistent = variants[variants > 1]

        if len(inconsistent) > 0:

            print(f"\n--- {col} ---")
            print(f"Inconsistent formatting groups: {len(inconsistent)}")

            for value in inconsistent.index:
                examples = (
                    series[normalized == value]
                    .drop_duplicates()
                    .tolist()
                )

                print(f"{value} -> {examples[:10]}")


DATASET: closed_deals

DATASET: customers

DATASET: geolocation

--- geolocation_city ---
Inconsistent formatting groups: 1
salvador -> ['salvador', 'salvador ']

DATASET: marketing_qualified_leads

DATASET: order_items

DATASET: order_payments

DATASET: order_reviews

--- review_comment_title ---
Inconsistent formatting groups: 496
10 -> ['10', ' 10', '10 ']
100 -> ['100', '100 ']
100% -> ['100%', '100% ']
4 -> [' 4 ', '4']
5 -> ['5', '5 ']
5 estrelas -> ['5 estrelas', '5 ESTRELAS', ' 5 estrelas']
7 -> ['7', '7 ']
acabamento -> ['acabamento', 'Acabamento']
acabamento ruim -> ['Acabamento Ruim', 'Acabamento ruim']
adorei -> ['Adorei', 'Adorei ', 'adorei']
aguardando -> ['Aguardando', 'Aguardando ']
ainda nao recebi -> ['Ainda nao recebi', 'ainda nao recebi']
ainda nao recebi o produt -> ['Ainda nao recebi o produt', 'Ainda NAO recebi o produt']
ainda não montei -> ['Ainda não montei ', 'ainda não montei']
ainda não recebi -> ['ainda não recebi ', 'Ainda não recebi ', 'ainda não recebi

In [33]:
# Check which variables are currently available
%whos

Variable                  Type           Data/Info
--------------------------------------------------
CLEAN_DATA_PATH           WindowsPath    ..\Data\Cleaned
Path                      type           <class 'pathlib.Path'>
RAW_DATA_PATH             WindowsPath    ..\Data\Raw
average_stock_numeric     Series         Shape: (842,)
categorical_checks        dict           n=9
categorical_cols          Index          Index(['product_category_<...>e_english'], dtype='str')
categorical_issues        list           n=8
categorical_issues_df     DataFrame      Shape: (8, 6)
closed                    DataFrame      Shape: (842, 14)
col                       str            product_category_name_english
column                    str            order_estimated_delivery_date
columns                   list           n=3
dataset                   str            closed_deals
dataset_name              str            product_category_name_translation
datasets                  dict           n=11
date_co

In [34]:
# Phase 6 - Numeric-like columns check

numeric_like_columns = {
    "closed_deals": [
        "average_stock",
        "declared_product_catalog_size",
        "declared_monthly_revenue"
    ],
    "products": [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ],
    "order_items": [
        "order_item_id",
        "price",
        "freight_value"
    ],
    "order_payments": [
        "payment_sequential",
        "payment_installments",
        "payment_value"
    ],
    "order_reviews": [
        "review_score"
    ]
}

for dataset, columns in numeric_like_columns.items():

    print("=" * 70)
    print(f"DATASET: {dataset}")
    print("=" * 70)

    df = datasets[dataset]

    for col in columns:
        if col in df.columns:
            print(f"\n{col}")
            print("Current dtype:", df[col].dtype)
            print("Non-null:", df[col].notna().sum())
            print("Sample values:", df[col].dropna().head(10).tolist())

DATASET: closed_deals

average_stock
Current dtype: str
Non-null: 66
Sample values: ['20-50', '20-50', '1-5', '1-5', '5-20', '5-20', '5-20', '200+', '1-5', '1-5']

declared_product_catalog_size
Current dtype: float64
Non-null: 69
Sample values: [2000.0, 80.0, 15.0, 120.0, 1000.0, 50.0, 4.0, 50.0, 400.0, 800.0]

declared_monthly_revenue
Current dtype: float64
Non-null: 842
Sample values: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
DATASET: products

product_name_lenght
Current dtype: float64
Non-null: 32341
Sample values: [40.0, 44.0, 46.0, 27.0, 37.0, 60.0, 56.0, 56.0, 57.0, 36.0]

product_description_lenght
Current dtype: float64
Non-null: 32341
Sample values: [287.0, 276.0, 250.0, 261.0, 402.0, 745.0, 1272.0, 184.0, 163.0, 1156.0]

product_photos_qty
Current dtype: float64
Non-null: 32341
Sample values: [1.0, 1.0, 1.0, 1.0, 4.0, 1.0, 4.0, 2.0, 1.0, 1.0]

product_weight_g
Current dtype: float64
Non-null: 32949
Sample values: [225.0, 1000.0, 154.0, 371.0, 625.0, 200.0, 18350.0, 

In [35]:
for dataset, col in [
    ("products", "product_width_cm"),
    ("order_items", "freight_value"),
    ("order_payments", "payment_value")
]:
    print("=" * 60)
    print(dataset, "→", col)
    print("dtype:", datasets[dataset][col].dtype)
    print("non-null:", datasets[dataset][col].notna().sum())
    print("sample:", datasets[dataset][col].dropna().head(10).tolist())

products → product_width_cm
dtype: float64
non-null: 32949
sample: [14.0, 20.0, 15.0, 26.0, 13.0, 11.0, 44.0, 40.0, 17.0, 12.0]
order_items → freight_value
dtype: float64
non-null: 112650
sample: [13.29, 19.93, 17.87, 12.79, 18.14, 12.69, 11.85, 70.75, 11.65, 11.4]
order_payments → payment_value
dtype: float64
non-null: 103886
sample: [99.33, 24.39, 65.71, 107.78, 128.45, 96.12, 81.16, 51.84, 341.09, 51.95]


In [36]:
# Phase 6 - ID column type check

id_columns = {
    "closed_deals": [
        "mql_id",
        "seller_id",
        "sdr_id",
        "sr_id"
    ],
    "customers": [
        "customer_id",
        "customer_unique_id"
    ],
    "marketing_qualified_leads": [
        "mql_id",
        "landing_page_id"
    ],
    "orders": [
        "order_id",
        "customer_id"
    ],
    "order_items": [
        "order_id",
        "product_id",
        "seller_id"
    ],
    "order_payments": [
        "order_id"
    ],
    "order_reviews": [
        "review_id",
        "order_id"
    ],
    "products": [
        "product_id"
    ],
    "sellers": [
        "seller_id"
    ]
}

for dataset, columns in id_columns.items():

    print("=" * 70)
    print(f"DATASET: {dataset}")
    print("=" * 70)

    df = datasets[dataset]

    for col in columns:
        if col in df.columns:
            print(f"{col}: {df[col].dtype}")

DATASET: closed_deals
mql_id: str
seller_id: str
sdr_id: str
sr_id: str
DATASET: customers
customer_id: str
customer_unique_id: str
DATASET: marketing_qualified_leads
mql_id: str
landing_page_id: str
DATASET: orders
order_id: str
customer_id: str
DATASET: order_items
order_id: str
product_id: str
seller_id: str
DATASET: order_payments
order_id: str
DATASET: order_reviews
review_id: str
order_id: str
DATASET: products
product_id: str
DATASET: sellers
seller_id: str


In [37]:
# Phase 6 - Final ID column check

for dataset_name, df in datasets.items():

    id_cols = [col for col in df.columns if "id" in col.lower()]

    if id_cols:
        print("=" * 70)
        print(f"DATASET: {dataset_name}")
        print("=" * 70)

        for col in id_cols:
            print(f"{col}: {df[col].dtype}")

DATASET: closed_deals
mql_id: str
seller_id: str
sdr_id: str
sr_id: str
DATASET: customers
customer_id: str
customer_unique_id: str
DATASET: marketing_qualified_leads
mql_id: str
landing_page_id: str
DATASET: order_items
order_id: str
order_item_id: int64
product_id: str
seller_id: str
DATASET: order_payments
order_id: str
DATASET: order_reviews
review_id: str
order_id: str
DATASET: orders
order_id: str
customer_id: str
DATASET: products
product_id: str
product_width_cm: float64
DATASET: sellers
seller_id: str


In [38]:
# Phase 6 - Standardize Boolean columns

closed = datasets["closed_deals"].copy()

closed["has_company"] = closed["has_company"].astype("boolean")
closed["has_gtin"] = closed["has_gtin"].astype("boolean")

datasets["closed_deals"] = closed

print("has_company:")
print(datasets["closed_deals"]["has_company"].dtype)

print("\nhas_gtin:")
print(datasets["closed_deals"]["has_gtin"].dtype)

has_company:
boolean

has_gtin:
boolean


In [39]:
# Phase 6 - Standardize average_stock

closed = datasets["closed_deals"].copy()

stock_mapping = {
    "1-5": 1,
    "5-20": 2,
    "20-50": 3,
    "50-200": 4,
    "200+": 5,
    "unknown": pd.NA
}

closed["average_stock_level"] = (
    closed["average_stock"]
    .map(stock_mapping)
    .astype("Int64")
)

datasets["closed_deals"] = closed

print(datasets["closed_deals"][
    ["average_stock", "average_stock_level"]
].value_counts(dropna=False))

average_stock  average_stock_level
NaN            <NA>                   776
5-20           2                       22
50-200         4                       15
1-5            1                       10
20-50          3                        8
200+           5                        7
unknown        <NA>                     4
Name: count, dtype: int64


In [40]:
# ============================================================
# STEP 12 — Numeric Field Review
# declared_monthly_revenue
# ============================================================

df = datasets["closed_deals"]

revenue = df["declared_monthly_revenue"]

print("Data type:", revenue.dtype)
print("Non-null:", revenue.notna().sum())
print("Missing:", revenue.isna().sum())
print("Zero values:", (revenue == 0).sum())
print("Positive values:", (revenue > 0).sum())
print("Negative values:", (revenue < 0).sum())

print("\nDescriptive statistics:")
print(revenue.describe())

print("\nPositive revenue values:")
print(revenue[revenue > 0].sort_values().to_string(index=False))

Data type: float64
Non-null: 842
Missing: 0
Zero values: 797
Positive values: 45
Negative values: 0

Descriptive statistics:
count    8.420000e+02
mean     7.337768e+04
std      1.744799e+06
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      5.000000e+07
Name: declared_monthly_revenue, dtype: float64

Positive revenue values:
       6.0
    1000.0
    4000.0
    5000.0
    5000.0
    6000.0
    8000.0
   10000.0
   10000.0
   10000.0
   15000.0
   15000.0
   20000.0
   20000.0
   20000.0
   25000.0
   25000.0
   25000.0
   30000.0
   30000.0
   30000.0
   40000.0
   50000.0
   50000.0
   60000.0
   60000.0
  100000.0
  100000.0
  100000.0
  100000.0
  100000.0
  120000.0
  120000.0
  130000.0
  150000.0
  180000.0
  200000.0
  210000.0
  250000.0
  250000.0
  300000.0
  300000.0
  500000.0
 8000000.0
50000000.0


In [41]:
# ============================================================
# STEP 12 — FINAL TRANSFORMATION CHECK
# ============================================================

print("CLOSED_DEALS COLUMNS")
print(datasets["closed_deals"].columns.tolist())

print("\n--- average_stock_level ---")
print(datasets["closed_deals"]["average_stock_level"].value_counts(dropna=False))

print("\n--- has_company ---")
print(datasets["closed_deals"]["has_company"].dtype)
print(datasets["closed_deals"]["has_company"].value_counts(dropna=False))

print("\n--- has_gtin ---")
print(datasets["closed_deals"]["has_gtin"].dtype)
print(datasets["closed_deals"]["has_gtin"].value_counts(dropna=False))

CLOSED_DEALS COLUMNS
['mql_id', 'seller_id', 'sdr_id', 'sr_id', 'won_date', 'business_segment', 'lead_type', 'lead_behaviour_profile', 'has_company', 'has_gtin', 'average_stock', 'business_type', 'declared_product_catalog_size', 'declared_monthly_revenue', 'average_stock_level']

--- average_stock_level ---
average_stock_level
<NA>    780
2        22
4        15
1        10
3         8
5         7
Name: count, dtype: Int64

--- has_company ---
boolean
has_company
<NA>     779
True      58
False      5
Name: count, dtype: Int64

--- has_gtin ---
boolean
has_gtin
<NA>     778
True      54
False     10
Name: count, dtype: Int64


In [42]:
# ============================================================
# STEP 12 — FINAL VALIDATION
# ============================================================

print("=" * 70)
print("STEP 12 — FINAL VALIDATION")
print("=" * 70)

# Closed deals transformations
closed = datasets["closed_deals"]

print("\n[1] average_stock_level")
print("dtype:", closed["average_stock_level"].dtype)
print("unique values:", closed["average_stock_level"].dropna().unique().tolist())
print("missing:", closed["average_stock_level"].isna().sum())

print("\n[2] has_company")
print("dtype:", closed["has_company"].dtype)
print("values:")
print(closed["has_company"].value_counts(dropna=False))

print("\n[3] has_gtin")
print("dtype:", closed["has_gtin"].dtype)
print("values:")
print(closed["has_gtin"].value_counts(dropna=False))

print("\n[4] declared_monthly_revenue")
revenue = closed["declared_monthly_revenue"]

print("dtype:", revenue.dtype)
print("missing:", revenue.isna().sum())
print("negative:", (revenue < 0).sum())
print("zero:", (revenue == 0).sum())
print("positive:", (revenue > 0).sum())

print("\n[5] datetime columns")

for dataset_name, columns in date_columns.items():
    df = datasets[dataset_name]

    for col in columns:
        if col in df.columns:
            print(
                f"{dataset_name} → {col}: "
                f"{df[col].dtype}"
            )

print("\n" + "=" * 70)
print("STEP 12 VALIDATION COMPLETE")
print("=" * 70)

STEP 12 — FINAL VALIDATION

[1] average_stock_level
dtype: Int64
unique values: [3, 1, 2, 5, 4]
missing: 780

[2] has_company
dtype: boolean
values:
has_company
<NA>     779
True      58
False      5
Name: count, dtype: Int64

[3] has_gtin
dtype: boolean
values:
has_gtin
<NA>     778
True      54
False     10
Name: count, dtype: Int64

[4] declared_monthly_revenue
dtype: float64
missing: 0
negative: 0
zero: 797
positive: 45

[5] datetime columns
closed_deals → won_date: datetime64[us]
marketing_qualified_leads → first_contact_date: datetime64[us]
order_items → shipping_limit_date: datetime64[us]
order_reviews → review_creation_date: datetime64[us]
order_reviews → review_answer_timestamp: datetime64[us]
orders → order_purchase_timestamp: datetime64[us]
orders → order_approved_at: datetime64[us]
orders → order_delivered_carrier_date: datetime64[us]
orders → order_delivered_customer_date: datetime64[us]
orders → order_estimated_delivery_date: datetime64[us]

STEP 12 VALIDATION COMPLETE


In [43]:
# ============================================================
# STEP 13 — FINAL STRUCTURAL VALIDATION
# ============================================================

print("=" * 70)
print("STEP 13 — FINAL STRUCTURAL VALIDATION")
print("=" * 70)

expected_shapes = {
    "closed_deals": (842, 15),
    "customers": (99441, 5),
    "geolocation": (1000163, 5),
    "marketing_qualified_leads": (8000, 4),
    "order_items": (112650, 7),
    "order_payments": (103886, 5),
    "order_reviews": (99224, 7),
    "orders": (99441, 8),
    "products": (32951, 9),
    "sellers": (3095, 4),
    "product_category_name_translation": (71, 2)
}

for name, df in datasets.items():

    expected = expected_shapes.get(name)
    actual = df.shape

    print(f"\n{name}")
    print(f"Expected shape : {expected}")
    print(f"Actual shape   : {actual}")

    if expected == actual:
        print("STATUS         : PASS")
    else:
        print("STATUS         : REVIEW")

print("\n" + "=" * 70)
print("STEP 13 STRUCTURAL VALIDATION COMPLETE")
print("=" * 70)

STEP 13 — FINAL STRUCTURAL VALIDATION

closed_deals
Expected shape : (842, 15)
Actual shape   : (842, 15)
STATUS         : PASS

customers
Expected shape : (99441, 5)
Actual shape   : (99441, 5)
STATUS         : PASS

geolocation
Expected shape : (1000163, 5)
Actual shape   : (1000163, 5)
STATUS         : PASS

marketing_qualified_leads
Expected shape : (8000, 4)
Actual shape   : (8000, 4)
STATUS         : PASS

order_items
Expected shape : (112650, 7)
Actual shape   : (112650, 7)
STATUS         : PASS

order_payments
Expected shape : (103886, 5)
Actual shape   : (103886, 5)
STATUS         : PASS

order_reviews
Expected shape : (99224, 7)
Actual shape   : (99224, 7)
STATUS         : PASS

orders
Expected shape : (99441, 8)
Actual shape   : (99441, 8)
STATUS         : PASS

products
Expected shape : (32951, 9)
Actual shape   : (32951, 9)
STATUS         : PASS

sellers
Expected shape : (3095, 4)
Actual shape   : (3095, 4)
STATUS         : PASS

product_category_name_translation
Expected 

In [44]:
# ============================================================
# STEP 14 — FINAL KEY / RELATIONSHIP VALIDATION
# ============================================================

print("=" * 70)
print("STEP 14 — FINAL KEY / RELATIONSHIP VALIDATION")
print("=" * 70)

# 1. Orders → Customers
orders = datasets["orders"]
customers = datasets["customers"]

invalid_customer_ids = ~orders["customer_id"].isin(customers["customer_id"])

print("\n[1] Orders → Customers")
print("Invalid customer IDs:", invalid_customer_ids.sum())


# 2. Order Items → Orders
order_items = datasets["order_items"]

invalid_order_ids = ~order_items["order_id"].isin(orders["order_id"])

print("\n[2] Order Items → Orders")
print("Invalid order IDs:", invalid_order_ids.sum())


# 3. Order Items → Products
products = datasets["products"]

invalid_product_ids = ~order_items["product_id"].isin(products["product_id"])

print("\n[3] Order Items → Products")
print("Invalid product IDs:", invalid_product_ids.sum())


# 4. Order Items → Sellers
sellers = datasets["sellers"]

invalid_seller_ids = ~order_items["seller_id"].isin(sellers["seller_id"])

print("\n[4] Order Items → Sellers")
print("Invalid seller IDs:", invalid_seller_ids.sum())


# 5. Order Payments → Orders
payments = datasets["order_payments"]

invalid_payment_order_ids = ~payments["order_id"].isin(orders["order_id"])

print("\n[5] Order Payments → Orders")
print("Invalid order IDs:", invalid_payment_order_ids.sum())


# 6. Order Reviews → Orders
reviews = datasets["order_reviews"]

invalid_review_order_ids = ~reviews["order_id"].isin(orders["order_id"])

print("\n[6] Order Reviews → Orders")
print("Invalid order IDs:", invalid_review_order_ids.sum())


# 7. Translation → Products categories
translation = datasets["product_category_name_translation"]

product_categories = set(
    products["product_category_name"].dropna().unique()
)

translation_categories = set(
    translation["product_category_name"].dropna().unique()
)

unmapped_translation_categories = translation_categories - product_categories

print("\n[7] Category Translation → Products")
print(
    "Translation categories not found in products:",
    len(unmapped_translation_categories)
)

print("\n" + "=" * 70)
print("STEP 14 KEY / RELATIONSHIP VALIDATION COMPLETE")
print("=" * 70)

STEP 14 — FINAL KEY / RELATIONSHIP VALIDATION

[1] Orders → Customers
Invalid customer IDs: 0

[2] Order Items → Orders
Invalid order IDs: 0

[3] Order Items → Products
Invalid product IDs: 0

[4] Order Items → Sellers
Invalid seller IDs: 0

[5] Order Payments → Orders
Invalid order IDs: 0

[6] Order Reviews → Orders
Invalid order IDs: 0

[7] Category Translation → Products
Translation categories not found in products: 0

STEP 14 KEY / RELATIONSHIP VALIDATION COMPLETE


In [45]:
# ============================================================
# SAVE CLEANED DATASETS
# ============================================================

from pathlib import Path

CLEAN_DATA_PATH = Path("../Data/Cleaned")
CLEAN_DATA_PATH.mkdir(parents=True, exist_ok=True)

for name, df in datasets.items():
    output_file = CLEAN_DATA_PATH / f"{name}.csv"
    df.to_csv(output_file, index=False)
    print(f"Saved: {output_file}")

print("\nAll cleaned datasets saved successfully.")

Saved: ..\Data\Cleaned\closed_deals.csv
Saved: ..\Data\Cleaned\customers.csv
Saved: ..\Data\Cleaned\geolocation.csv
Saved: ..\Data\Cleaned\marketing_qualified_leads.csv
Saved: ..\Data\Cleaned\order_items.csv
Saved: ..\Data\Cleaned\order_payments.csv
Saved: ..\Data\Cleaned\order_reviews.csv
Saved: ..\Data\Cleaned\orders.csv
Saved: ..\Data\Cleaned\products.csv
Saved: ..\Data\Cleaned\sellers.csv
Saved: ..\Data\Cleaned\product_category_name_translation.csv

All cleaned datasets saved successfully.


In [46]:
# ============================================================
# VERIFY CLEANED FILES
# ============================================================

cleaned_files = sorted(CLEAN_DATA_PATH.glob("*.csv"))

print("Number of cleaned files:", len(cleaned_files))
print("\nCleaned files:")

for file in cleaned_files:
    print("-", file.name)

Number of cleaned files: 11

Cleaned files:
- closed_deals.csv
- customers.csv
- geolocation.csv
- marketing_qualified_leads.csv
- order_items.csv
- order_payments.csv
- order_reviews.csv
- orders.csv
- product_category_name_translation.csv
- products.csv
- sellers.csv


In [47]:
# ============================================================
# VERIFY SAVED TRANSFORMATIONS
# ============================================================

saved_closed = pd.read_csv(
    CLEAN_DATA_PATH / "closed_deals.csv"
)

print("closed_deals columns:")
print(saved_closed.columns.tolist())

print("\naverage_stock_level dtype:")
print(saved_closed["average_stock_level"].dtype)

print("\naverage_stock_level values:")
print(saved_closed["average_stock_level"].value_counts(dropna=False))

print("\nhas_company dtype:")
print(saved_closed["has_company"].dtype)

print("\nhas_gtin dtype:")
print(saved_closed["has_gtin"].dtype)

closed_deals columns:
['mql_id', 'seller_id', 'sdr_id', 'sr_id', 'won_date', 'business_segment', 'lead_type', 'lead_behaviour_profile', 'has_company', 'has_gtin', 'average_stock', 'business_type', 'declared_product_catalog_size', 'declared_monthly_revenue', 'average_stock_level']

average_stock_level dtype:
float64

average_stock_level values:
average_stock_level
NaN    780
2.0     22
4.0     15
1.0     10
3.0      8
5.0      7
Name: count, dtype: int64

has_company dtype:
object

has_gtin dtype:
object


In [48]:
# ============================================================
# FINAL CHECK — SAVED CLEANED DATASETS
# ============================================================

print("=" * 70)
print("FINAL CHECK — SAVED CLEANED DATASETS")
print("=" * 70)

for name, expected_shape in expected_shapes.items():

    file_path = CLEAN_DATA_PATH / f"{name}.csv"

    saved_df = pd.read_csv(file_path)

    print(
        f"{name:<40} "
        f"{saved_df.shape} "
        f"{'PASS' if saved_df.shape == expected_shape else 'REVIEW'}"
    )

print("\n" + "=" * 70)
print("SAVED DATASET VALIDATION COMPLETE")
print("=" * 70)

FINAL CHECK — SAVED CLEANED DATASETS
closed_deals                             (842, 15) PASS
customers                                (99441, 5) PASS
geolocation                              (1000163, 5) PASS
marketing_qualified_leads                (8000, 4) PASS
order_items                              (112650, 7) PASS
order_payments                           (103886, 5) PASS
order_reviews                            (99224, 7) PASS
orders                                   (99441, 8) PASS
products                                 (32951, 9) PASS
sellers                                  (3095, 4) PASS
product_category_name_translation        (71, 2) PASS

SAVED DATASET VALIDATION COMPLETE
